In [1]:
print("Hi")

Hi


In [ ]:
import os
from dotenv import load_dotenv 
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS

In [2]:
load_dotenv()

True

In [3]:
groq_key=os.getenv("GROQ_API_KEY")
jina_key=os.getenv("JINA_API_KEY")



In [4]:
DATA_FILE_PATH=os.path.join("data","hr_policy.txt")

In [6]:
loader= TextLoader(DATA_FILE_PATH,encoding="utf-8")
documents=loader.load()
print(documents)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probatio

In [8]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
     
)
chunks=text_splitter.split_documents(documents)
print(chunks)

[Document(metadata={'source': 'data/hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data/hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to

In [11]:
embeddings_model=JinaEmbeddings(model_name=("jina-embeddings-v2-base-en"))

In [13]:
vector_store=FAISS.from_documents(chunks,embeddings_model)


In [14]:
test_query="How many sick leaves do employees get?"

top_matches=vector_store.similarity_search(test_query,k=2)

In [15]:
for i, match in enumerate(top_matches,start=1):
    print("Match ",i)
    print(match.page_content)
    print()

Match  1
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

Match  2
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



In [29]:
retriever=vector_store.as_retriever(search_kwargs={"k":3})
def search_hr_policy(question:str)->str:
    matching_chunks=retriever.invoke(question)
    return "\n\n".join(chunk.page_content for chunk in matching_chunks)

In [ ]:
from langchain_groq import ChatGroq

llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1
)

llm.model_name

'openai/gpt-oss-120b'

In [18]:
test_response=llm.invoke("Hey what's up")
print(test_response.content)

Hey there! I'm doing great—thanks for asking. How can I help you today?


In [ ]:
from langchain.agents import create_agent
hr_assistant=create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt=""" 
"You are a friendly HR assistant. Always use seach_hr_policy tool to look up" 
"Facts before answering, If the answer is not in the search result say you do not know rather than guessing" 
"""
)

In [20]:
def ask_hr_assistant(question:str)->str:
    print("="*60)
    print("Question : ",question)
    print("="*60)

    response=hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer=response["messages"][-1].content

    print("Answer : ",answer)
    print("="*60)
    print()
    return answer


In [26]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"Hi bro whats up"

            }
        ]
    }
)

In [28]:
response["messages"][-1].content

'Hey there! I’m doing great—thanks for asking. How can I help you today? If you have any HR‑related questions, just let me know!'